## Scaricare il dataset

* **Scaricare il dataset [MNIST](https://en.wikipedia.org/wiki/MNIST_database). Questo dataset è composto da 60k immagini a scala di grigio di dimensione 28x28 di numeri scritti a mano.**


*Si sottolinea che i commenti al laboratorio sono riferiti ad una nostra esecuzione del codice ma a nuove esecuzioni potrebbero variare e  potrebbero risultare imprecisi, senza perdere tuttavia di validità generale.*

In [ ]:
#Import delle librerie
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Dropout

from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import numpy as np
from numpy import array
import matplotlib.pyplot as plt

#Definisco un seed fissato, in modo da avere riproducibilità dei risultati senza variazioni e casualità
seed = 42

#Seed per numpy e per tensorflow
np.random.seed(seed)
tf.random.set_seed(seed)
from pathlib import Path
ARTIFACTS = Path("artifacts/mnist")
ARTIFACTS.mkdir(parents=True, exist_ok=True)


## Visualizzare ed esplorare il dataset

* **Plottare nello stesso grafico un’immagine per classe, inserendo come titolo della singola immagine l'etichetta corrispondente.**

In [ ]:
#Load dataset
(trainX, trainy), (testX, testy) = mnist.load_data()

In [ ]:
#Vettore dei valori unici dei label del Training Set
unique_label = set(trainy)
print("Il dataset fornito ha un totale di "+str(len(unique_label))+" classi.")

#Conteggio di immagini per ogni classe
for label in unique_label:
  num = np.shape(np.where(trainy==label))
  tot=np.shape(trainy)
  perc=round(num[1]*100/tot[0],2)
  print("Numero di immagini labellate nella classe "+str(label)+": "+str(num[1])+" e si presenta per "+str(perc)+"% del campione")

In [ ]:
#Visualizzazione di un esempio per classe
plt.figure(figsize=(10, 10))
for label in unique_label:
  idx=np.where(trainy==label)
  ax=plt.subplot(4, 5, label + 1)
  plt.imshow(trainX[idx[0][0]].astype("uint8"))
  plt.title("Classe "+ str(label))
  plt.axis("off")
plt.subplots_adjust(wspace=0.1,hspace=0)

* **Verificare la distribuzione delle classi del dataset plottando un istogramma**

In [ ]:
#Istogramma della distrubuzione del dataset
counts = np.bincount(trainy)
fig,ax = plt.subplots()
ax.bar(range(10),counts,width=0.8,align='center');
ax.set(xticks=range(10),xlim=[-1,10]);
plt.title("Distribuzione del dataset");
plt.xlabel('Labels');
plt.ylabel('Numero di campioni');
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show();

Si può osservare dall'istogramma che sono presenti 10 classi: cifre di numeri interi tra 0 e 9; in particolare, la cifra più frequente è 1, la quale ricopre l'11.24% del totale dei campioni, mentre la cifra meno frequente è 5, il 9.04% dei campioni.

Tuttavia, il dataset risulta bilanciato quindi ci si aspetta che il modello non si specializzi su una classe in particolare.

* **Considerare solamente le prime 4000 immagini per costruire il dataset dell'esperimento.**

In [ ]:
#Dataset ridotto di soli 4000 campioni
data_X,data_y = trainX[0:4000],trainy[0:4000]
data_X = data_X[..., np.newaxis]


L'osservazione fatta precedentemente sull'uniformità del dataset era in riferimento a tutte le 60000 immagini, non è detto che valga la stessa osservazione sui soli 4000 campioni, pertanto studiamo nuovamente l'istogramma nel caso del dataset ridotto.

In [ ]:
#Conteggio di immagini per ogni classe
for label in unique_label:
  num = np.shape(np.where(data_y==label))
  tot=np.shape(data_y)
  perc=round(num[1]*100/tot[0],2)
  print("Numero di immagini labellate nella classe "+str(label)+": "+str(num[1])+" e si presenta per "+str(perc)+"% del campione")


In [ ]:
#Istogramma della distrubuzione del dataset ridotto
counts = np.bincount(data_y)
fig,ax = plt.subplots()
ax.bar(range(10),counts,width=0.8,align='center');
ax.set(xticks=range(10),xlim=[-1,10]);
plt.title("Distribuzione del dataset ridotto");
plt.xlabel('Labels');
plt.ylabel('Numero di campioni');
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show();

E' possibile osservare, nuovamente, che la classe 1 è la più frequente (circa l'11.1% del totale dei campioni come prima) e che la classe 5 è sempre la meno frequente (di nuovo, circa lo 8.75% dei campioni); tuttavia anche il dataset ridotto risulta ben bilanciato.

* **Convertire le etichette del dataset in maniera appopriata (*one-hot encoding*).**

In [ ]:
#One-hot encoding (per classificazione multi-classe)
data_y = to_categorical(data_y)

#Test
plt.imshow(data_X[0].squeeze())
print('La rispettiva etichetta one-hot encoded è ', data_y[0])
#La prima immagine del dataset è un 5, la rispettiva etichetta presenta 1 in quinta posizione

## Training del modello base CNN

* **Per addestrare il modello, utilizziamo la [5-fold crossvalidation](https://machinelearningmastery.com/k-fold-cross-validation/), integrando la parte di pre-processing e data augmentation come abbiamo visto nel laboratorio sulla Classificazione di immagini. Visualizzare il sommario del modello prima di addestrarlo.**

* **Verificare la performance di classificazione al variare degli split.**

* **Plottare grafici per validation per ogni split. Sull'asse delle *x* indichiamo il numero delle epoche, sull'asse *y* indichiamo invece l'accuratezza. In arancione plottiamo la curva di training, in blu quella di test. Inserire la legenda nel grafico, e per ogni grafico inserire il titolo con indicato lo split considerato.**

* **Calcolare e visualizzare le matrici di confusione per ogni split.**


* **Per ogni split, visualizzare esempi di immagini di test con titolo "GT: X, PRED: Y", dove *GT* è la classe vera, mentre *PRED* è la previsione della rete.**

**TRAINING CON DATA AUGMENTATION POCO EFFICACE**

Abbiamo inserito un flip verticale ed orizzontale, delle rotazioni di fattore 0.25 e, come atteso, il risultato non è soddisfacente perchè:


*   **Flip verticale:** le cifre 2 e 5 potrebbero essere confuse,
*   **Flip orizzontale:** le cifre 6 e 9 potrebbero essere confuse,
*   **Rotazione:** è innaturale riconoscere le cifre tanto ruotate.

Il modello utilizzato in questa fase prevede il layer di data augmentation, un layer di normalizzazione, un layer di convoluzione con Kernel 3x3, uno di Max Pooling con Kernel 2x2, aggiornamento dei pesi con SGD e learning rate fisso a 0.01.

I parametri della rete sono stati plottati di seguito.

In [ ]:
#Preprocessing

#Definizione del modello
def define_model():
 data_augmentation = keras.Sequential(
   [
     layers.RandomFlip("horizontal",
                       input_shape=(28,
                                   28,
                                   1)),
     layers.RandomFlip("vertical",
                   input_shape=(28,
                               28,
                               1)),
     layers.RandomRotation(0.25),
     layers.RandomZoom(0.25),
   ]
 )
 
 model = Sequential()
 model.add(keras.Input(shape=(28,28,1)))
 model.add(data_augmentation)
 model.add(layers.Rescaling(1./255))
 model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', input_shape=(28, 28, 1)))
 model.add(MaxPooling2D((2, 2)))
 model.add(Flatten())
 model.add(Dense(100, activation='relu', kernel_initializer='he_uniform'))
 model.add(Dense(10, activation='softmax'))
 # compile model
 opt = SGD(learning_rate=0.01, momentum=0.9)
 model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
 return model

In [ ]:
#Creazione del modello
model = define_model()

#Visualizzazione del sommario del modello
model.summary()

In [ ]:
#Inizializzazione cross validation
epochs=100
kfold = KFold(n_splits=5)

#Inizializzazione statistiche
accuracy_per_fold = []
loss_per_fold = []
val_accuracy_per_fold = []
val_loss_per_fold = []

#Ciclo su ogni split
for fold, (idx_train, idx_val) in enumerate(kfold.split(data_X)):
  model = define_model()
  #Splitto in insieme di training e insieme di validation
  train_X_cv = data_X[idx_train]
  train_y_cv = data_y[idx_train]
  val_X_cv = data_X[idx_val]
  val_y_cv = data_y[idx_val]


  checkpoint_filepath = str(ARTIFACTS / ('model_'+str(fold)+'.keras'))
  model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)
  print("Inizio allenamento split numero: "+str(fold+1))
  print("")
  history = model.fit(train_X_cv,train_y_cv, validation_data=[val_X_cv,val_y_cv], epochs=epochs, callbacks=[model_checkpoint_callback])
  print("")
  print("Fine allenamento split numero: "+str(fold+1))
  print("")

  #Accuracy e Loss del modello
  acc = history.history['accuracy']
  val_acc = history.history['val_accuracy']

  loss = history.history['loss']
  val_loss = history.history['val_loss']

  #Plot performance modello
  plt.figure(figsize=(10, 5))
  plt.subplot(1, 2, 1)
  plt.plot(range(0,len(val_acc)), val_acc, label='Validation Accuracy')
  plt.plot(range(0,len(acc)), acc, label='Training Accuracy')
  plt.legend(loc='lower right')
  plt.title('Training and Validation Accuracy (split numero '+str(fold+1)+')')

  plt.subplot(1, 2, 2)
  plt.plot(range(0,len(val_loss)), val_loss, label='Validation Loss')
  plt.plot(range(0,len(loss)), loss, label='Training Loss')
  plt.legend(loc='upper right')
  plt.title('Training and Validation Loss (split numero '+str(fold+1)+')')
  plt.tight_layout()
  plt.show()
  print("-------------------------------------------------------------------------")

  print("-------------------------------------------------------------------------")
  print("MATRICE DI CONFUSIONE:")
  print("-------------------------------------------------------------------------")

  model = keras.models.load_model(checkpoint_filepath)
  selected_loss, selected_acc = model.evaluate(train_X_cv, train_y_cv, verbose=0)
  selected_val_loss, selected_val_acc = model.evaluate(val_X_cv, val_y_cv, verbose=0)

  #Etichette predette dal modello sul validation
  val_predict = model.predict(val_X_cv)           #Per ogni immagine si ottiene un vettore con le probabilità di appartenere a ciascuna classe, da 0 a 9
  val_predict = np.argmax(val_predict, axis=1)    #Considero l'indice dell'elemento che presenta probabilità più alta
  #Etichette reali del validation
  val_y_cv_labels = np.argmax(val_y_cv, axis=1)   #Siccome è in forma one-hot encoded presenta solo 0 e 1. Prendo indice dell'elemento massimo cioè 1


  cm = confusion_matrix(val_y_cv_labels, val_predict)
  #Plot matrice di confusione
  disp = ConfusionMatrixDisplay(confusion_matrix=cm)
  disp.plot(cmap=plt.cm.Blues)
  plt.title('Confusion Matrix (split numero '+str(fold+1)+')')
  plt.show()

  #Calcolo statistiche
  val_accuracy_per_fold.append(selected_val_acc)
  val_loss_per_fold.append(selected_val_loss)
  accuracy_per_fold.append(selected_acc)
  loss_per_fold.append(selected_loss)


  #Immagini di Test sul Validation Set
  print('----------------------------------------------------------------------------------------------------------------')
  plt.figure(figsize=(10, 10))
  for i in range(10):
      plt.subplot(4, 5, i + 1)
      plt.imshow(val_X_cv[i].squeeze(), cmap="gray")
      plt.axis('off')
      plt.title(f'GT: {val_y_cv_labels[i]}, PRED: {val_predict[i]}')
  plt.suptitle(f'Esempio di immagini del Validation (split numero {fold+1})')
  plt.tight_layout()
  plt.show()

* **Calcolare la performance media degli split e la deviazione standard per avere una metrica globale con cui valutare il modello.**

In [ ]:
#Considero gli array di accuracy/loss riempiti precedentemente in base a training e validation per ogni split
#Ne faccio media e deviazione standard

mean_accuracy = np.mean(accuracy_per_fold)
std_accuracy = np.std(accuracy_per_fold)

mean_val_accuracy = np.mean(val_accuracy_per_fold)
std_val_accuracy = np.std(val_accuracy_per_fold)

mean_loss = np.mean(loss_per_fold)
std_loss = np.std(loss_per_fold)

mean_val_loss = np.mean(val_loss_per_fold)
std_val_loss = np.std(val_loss_per_fold)

print("-------------------------------------------------------------------------")
print("Prestazioni globali del modello:")
print("-------------------------------------------------------------------------")
print(f"Accuratezza media sul Training Set: {mean_accuracy:.4f} (± {std_accuracy:.4f})")
print(f"Perdita media sul Training Set: {mean_loss:.4f} (± {std_loss:.4f})")
print("-------------------------------------------------------------------------")
print(f"Accuratezza media sul Validation Set: {mean_val_accuracy:.4f} (± {std_val_accuracy:.4f})")
print(f"Perdita media sul Validation Set: {mean_val_loss:.4f} (± {std_val_loss:.4f})")
print("-------------------------------------------------------------------------")

Results are generated by the preceding cells. Compare validation metrics and confusion matrices; saved historical scores are omitted because the selected-checkpoint evaluation differs from the original experiment.

Training and validation losses should both be inspected for overfitting. Fluctuations alone do not measure statistical bias. Final conclusions require held-out evaluation.

## Come migliorare la performance

** Dopo aver ottenuto una performance di base nell'esperimento precedente, cercare di migliorare la performance ottenuta. Ad esempio:**

* **   Provare ad aumentare il numero di filtri**
* **  Provare ad aumentare il numero di layers**
* **  Inserire la regolarizzazione (dropout, early stopping, ...)**
* **  Bilanciare le classi ([class weights](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#calculate_class_weights)) **
* **  Altro **




The revised CNN removes flips, reduces augmentation strength, uses a deeper convolutional structure with fewer total parameters, adds dropout and uses Adam. Several components change at once; this is not an ablation study.

Per velocizzare il nostro modello, abbiamo inserito nelle callbacks l'Early Stopping con un termine di pazienza di 35 epoche, e la Riduzione del Learning Rate con pazienza pari a 15 epoche.

Il numero totale di epoche è stato aumentato a 300 al fine di poter migliorare le performance e poter notare l'effetto dell'Early Stopping dovuto ad un eventuale overfitting sul Validation Set.

In [ ]:
#Preprocessing

#Definizione del modello migliorato
def define_better_model():
 data_augmentation = keras.Sequential(
   [
     layers.RandomRotation((-0.1, 0.1), input_shape=(28,28,1)),
     layers.RandomZoom((-0.2, 0.2), input_shape=(28,28,1))
   ]
 )
 
 model = Sequential()
 model.add(keras.Input(shape=(28,28,1)))
 model.add(data_augmentation)
 model.add(layers.Rescaling(1./255))
 model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', input_shape=(28,28,1)))
 model.add(MaxPooling2D((2, 2)))
 model.add(Dropout(0.2))
 model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
 model.add(MaxPooling2D((2, 2)))
 model.add(Conv2D(128, (2, 2), activation='relu', kernel_initializer='he_uniform', input_shape=(28,28,1)))
 model.add(MaxPooling2D((2, 2)))
 model.add(Dropout(0.2))
 model.add(Flatten())
 model.add(Dense(64, activation='relu', kernel_initializer='he_uniform'))
 model.add(Dense(10, activation='softmax'))
 # compile model
 #opt = SGD(learning_rate=0.001, momentum=0.9)
 #non usiamo stochastic gradient descent ma adam per avere prestazioni migliori
 model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 return model

In [ ]:
#Creazione del modello
better_model = define_better_model()

#Visualizzazione del sommario del modello
better_model.summary()

In [ ]:
#Inizializzazione cross validation
epochs=300
kfold = KFold(n_splits=5)

#Inizializzazione statistiche
better_accuracy_per_fold = []
better_loss_per_fold = []
better_val_accuracy_per_fold = []
better_val_loss_per_fold = []


#Ciclo su ogni split
for fold, (idx_train, idx_val) in enumerate(kfold.split(data_X)):
  better_model = define_better_model()
  #Splitto in insieme di training e insieme di validation
  train_X_cv = data_X[idx_train]
  train_y_cv = data_y[idx_train]

  val_X_cv = data_X[idx_val]
  val_y_cv = data_y[idx_val]


  checkpoint_filepath = str(ARTIFACTS / ('better_model_'+str(fold)+'.keras'))
  callbacks = [
    keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath, monitor='val_accuracy', mode='max', save_best_only=True),
    #Aggiungiamo un early stopping, se la validation loss non diminuisce dopo 35 epoche allora ci fermiamo
    keras.callbacks.EarlyStopping(monitor = 'val_loss',mode = 'min', patience = 35, verbose = 1),
    #Aggiungiamo una riduzione del passo learning rate
    keras.callbacks.ReduceLROnPlateau(monitor = 'val_loss',factor = 0.8, patience = 15, verbose = 1)
  ]

  print("Inizio allenamento split numero: "+str(fold+1))
  print("")
  history = better_model.fit(train_X_cv,train_y_cv, validation_data=[val_X_cv,val_y_cv], epochs=epochs, callbacks=callbacks)
  print("")
  print("Fine allenamento split numero: "+str(fold+1))
  print("")

  #Accuracy e Loss del modello
  acc = history.history['accuracy']
  val_acc = history.history['val_accuracy']

  loss = history.history['loss']
  val_loss = history.history['val_loss']

  epochs_range = range(epochs)

  #Plot performance modello
  plt.figure(figsize=(10, 5))
  plt.subplot(1, 2, 1)
  plt.plot(range(0,len(val_acc)), val_acc, label='Validation Accuracy')
  plt.plot(range(0,len(acc)), acc, label='Training Accuracy')
  plt.legend(loc='lower right')
  plt.title('Training and Validation Accuracy (split numero '+str(fold+1)+')')

  plt.subplot(1, 2, 2)
  plt.plot(range(0,len(val_loss)), val_loss, label='Validation Loss')
  plt.plot(range(0,len(loss)), loss, label='Training Loss')
  plt.legend(loc='upper right')
  plt.title('Training and Validation Loss (split numero '+str(fold+1)+')')
  plt.tight_layout()
  plt.show()
  print("-------------------------------------------------------------------------")

  print("-------------------------------------------------------------------------")
  print("MATRICE DI CONFUSIONE:")
  print("-------------------------------------------------------------------------")

  better_model = keras.models.load_model(checkpoint_filepath)
  selected_loss, selected_acc = better_model.evaluate(train_X_cv, train_y_cv, verbose=0)
  selected_val_loss, selected_val_acc = better_model.evaluate(val_X_cv, val_y_cv, verbose=0)

  #Etichette predette dal modello sul validation
  val_predict = better_model.predict(val_X_cv)    #Per ogni immagine si ottiene un vettore con le probabilità di appartenere a ciascuna classe, da 0 a 9
  val_predict = np.argmax(val_predict, axis=1)    #Considero l'indice dell'elemento che presenta probabilità più alta
  #Etichette reali del validation
  val_y_cv_labels = np.argmax(val_y_cv, axis=1)   #Siccome è in forma one-hot encoded presenta solo 0 e 1. Prendo indice dell'elemento massimo cioè 1


  cm = confusion_matrix(val_y_cv_labels, val_predict)
  #Plot matrice di confusione
  disp = ConfusionMatrixDisplay(confusion_matrix=cm)
  disp.plot(cmap=plt.cm.Blues)
  plt.title('Confusion Matrix (split numero '+str(fold+1)+')')
  plt.show()

  #Calcolo statistiche
  better_val_accuracy_per_fold.append(selected_val_acc)
  better_val_loss_per_fold.append(selected_val_loss)
  better_accuracy_per_fold.append(selected_acc)
  better_loss_per_fold.append(selected_loss)


  #Immagini di Test sul Validation
  print('----------------------------------------------------------------------------------------------------------------')
  plt.figure(figsize=(10, 10))
  for i in range(10):
      plt.subplot(4, 5, i + 1)
      plt.imshow(val_X_cv[i].squeeze(), cmap="gray")
      plt.axis('off')
      plt.title(f'GT: {val_y_cv_labels[i]}, PRED: {val_predict[i]}')
  plt.suptitle(f'Esempio di immagini del Validation (split numero {fold+1})')
  plt.tight_layout()
  plt.show()

Results are generated by the preceding cells. Compare validation metrics and confusion matrices; saved historical scores are omitted because the selected-checkpoint evaluation differs from the original experiment.

In [ ]:
#Considero gli array di accuracy/loss riempiti precedentemente in base a training e validation per ogni split
#Ne faccio media e deviazione standard

better_mean_accuracy = np.mean(better_accuracy_per_fold)
better_std_accuracy = np.std(better_accuracy_per_fold)

better_mean_val_accuracy = np.mean(better_val_accuracy_per_fold)
better_std_val_accuracy = np.std(better_val_accuracy_per_fold)

better_mean_loss = np.mean(better_loss_per_fold)
better_std_loss = np.std(better_loss_per_fold)

better_mean_val_loss = np.mean(better_val_loss_per_fold)
better_std_val_loss = np.std(better_val_loss_per_fold)

print("-------------------------------------------------------------------------")
print("Prestazioni globali del modello:")
print("-------------------------------------------------------------------------")
print(f"Accuratezza media sul training: {better_mean_accuracy:.4f} (± {better_std_accuracy:.4f})")
print(f"Perdita media sul training: {better_mean_loss:.4f} (± {better_std_loss:.4f})")
print("-------------------------------------------------------------------------")
print(f"Accuratezza media sul validation: {better_mean_val_accuracy:.4f} (± {better_std_val_accuracy:.4f})")
print(f"Perdita media sul validation: {better_mean_val_loss:.4f} (± {better_std_val_loss:.4f})")
print("-------------------------------------------------------------------------")

Results are generated by the preceding cells. Compare validation metrics and confusion matrices; saved historical scores are omitted because the selected-checkpoint evaluation differs from the original experiment.

In [ ]:
delta_acc = np.array(better_val_accuracy_per_fold) - np.array(val_accuracy_per_fold)
print("Paired accuracy differences (percentage points):", 100 * delta_acc)
print("Mean / SD (not a confidence interval):", 100*delta_acc.mean(), 100*delta_acc.std())


Results are generated by the preceding cells. Compare validation metrics and confusion matrices; saved historical scores are omitted because the selected-checkpoint evaluation differs from the original experiment.

In [ ]:
print("Relative validation loss reduction (%):", 100 * (mean_val_loss - better_mean_val_loss) / mean_val_loss)


Results are generated by the preceding cells. Compare validation metrics and confusion matrices; saved historical scores are omitted because the selected-checkpoint evaluation differs from the original experiment.

## Opzionali

### PCA & Classificatore standard

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier


Ridimensionare le immagini di training e di test in vettori 1D, utilizzando

```
trainX.reshape
```



Standardizzare i dati utilizzando [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) . In particolare, fittare la trasformazione sui dati di training, e applicarla (`transform`) anche su quelli di test.



In [ ]:
train_flat = data_X.reshape(len(data_X), -1).astype(np.float32)
test_flat = testX.reshape(len(testX), -1).astype(np.float32)
train_labels = np.argmax(data_y, axis=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_flat)
X_test_scaled = scaler.transform(test_flat)


Utilizzare la [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) per ridurre la dimensionalità. Partire con numero di componenti tali da mantenere il 95% della varianza, e poi fare prove al variare di tale valore.


In [ ]:
pca = PCA(n_components=0.95, svd_solver="full")


Fittare la trasformazione sui dati di training e applicarla anche ai dati di test.

In [ ]:
trainX_pca = pca.fit_transform(X_train_scaled)
testX_pca = pca.transform(X_test_scaled)
print("PCA components:", trainX_pca.shape[1])


Utilizzare le componenti principali come feature in un classificatore standard, ad esempio [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html). Provare a variare i parametri del modello.

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(trainX_pca, train_labels)


Fare una previsione del modello sui dati di test

In [ ]:
from sklearn.metrics import accuracy_score
rf_prediction = rf_clf.predict(testX_pca)
print("PCA/RF training accuracy:", rf_clf.score(trainX_pca, train_labels))
print("PCA/RF official test accuracy:", accuracy_score(testy, rf_prediction))


Valutare l'accuratezza per training e test set


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12,5))
for i, ax in enumerate(axes.flat):
    ax.imshow(testX[i], cmap="gray"); ax.set_title(f"GT {testy[i]}, PRED {rf_prediction[i]}"); ax.axis("off")
plt.tight_layout(); plt.show()


Plottare in un grafico le immagini di test, e nel titolo inserire informazione sulla classe vera e su quella predetta dal modello.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
rf_pipeline = make_pipeline(StandardScaler(), PCA(n_components=0.95, svd_solver="full"),
                            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
# Fit scaling and PCA separately inside each training fold (no preprocessing leakage).
rf_cv = cross_val_score(rf_pipeline, train_flat, train_labels, cv=kfold, scoring="accuracy")
print("PCA/RF cross-validation:", rf_cv.mean(), rf_cv.std())


Effettuare lo stesso tipo di analisi in modalità di cross-validazione, come nell'esempio precedente.

### Classificatore ibrido

In questa sezione, si utilizzerà la parte di estrazione delle feature della CNN in combinazione ad un classificatore standard (es. Random Forest).


1) Caricare il modello pre-addestrato e rimuovere lo strato di classificazione

In [ ]:
# Dedicated trained CNN for feature extraction: do not use an untrained network.
hybrid_train, hybrid_val = train_test_split(np.arange(len(data_X)), test_size=0.2,
                                          stratify=train_labels, random_state=42)
loaded_model = define_better_model()
loaded_model.fit(data_X[hybrid_train], data_y[hybrid_train],
                 validation_data=(data_X[hybrid_val], data_y[hybrid_val]), epochs=100, verbose=2,
                 callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                                         restore_best_weights=True)])


In [ ]:
layer_output = loaded_model.layers[-2].output


Creare un modello che dato l'input fornisca l'output desiderato

In [ ]:
act_model = keras.Model(inputs=loaded_model.inputs, outputs=layer_output)
act_model.summary()


Applicare il modello ai dati di training e di test per ottenere le feature di interesse

In [ ]:
X_train_feat = act_model.predict(data_X[hybrid_train], verbose=0)
X_val_feat = act_model.predict(data_X[hybrid_val], verbose=0)
X_test_feat = act_model.predict(testX[..., np.newaxis], verbose=0)


Effettua l'addestramento con Random Forest e compara i risultati ottenuti con quelli precedenti.

In [ ]:
hybrid_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
hybrid_rf.fit(X_train_feat, train_labels[hybrid_train])
print("Hybrid validation accuracy:", hybrid_rf.score(X_val_feat, train_labels[hybrid_val]))
print("Hybrid official test accuracy:", hybrid_rf.score(X_test_feat, testy))


#### GradCAM (avanzato)

Provare a seguire il tutorial su [gradcam](https://pyimagesearch.com/2020/03/09/grad-cam-visualize-class-activation-maps-with-keras-tensorflow-and-deep-learning/). Questa tecnica serve per cercare di interpretare i risultati di classificazione ottenuti, evidenziando le regioni che hanno maggiormente contribuito alla decisione del modello.

In [ ]:
# Grad-CAM explains the CNN output, not the Random Forest.
last_conv = next(layer for layer in reversed(loaded_model.layers) if isinstance(layer, Conv2D))
grad_model = keras.Model(loaded_model.inputs, [last_conv.output, loaded_model.output])
image = tf.convert_to_tensor(testX[:1, ..., np.newaxis], dtype=tf.float32)
with tf.GradientTape() as tape:
    activations, probabilities = grad_model(image, training=False)
    predicted_class = tf.argmax(probabilities[0])
    score = probabilities[:, predicted_class]
gradients = tape.gradient(score, activations)
weights = tf.reduce_mean(gradients, axis=(0,1,2))
heatmap = tf.maximum(tf.reduce_sum(activations[0] * weights, axis=-1), 0)
heatmap /= tf.reduce_max(heatmap) + keras.backend.epsilon()
heatmap = tf.image.resize(heatmap[..., None], (28,28)).numpy().squeeze()
plt.imshow(testX[0], cmap="gray"); plt.imshow(heatmap, cmap="jet", alpha=0.45)
plt.title(f"CNN Grad-CAM: class {int(predicted_class)}"); plt.axis("off"); plt.show()
